In [1]:
# 06_heloc_missing.ipynb
# Third dataset: FICO HELOC (10,459 rows, 52% bad, 23 numeric features).
# Focus: SPECIAL MISSING CODES -7/-8/-9 are INFORMATIVE, not blanks
#   -9 = no bureau record, -8 = no usable/under investigation, -7 = condition not met
#   (e.g. MSinceMostRecentDelq=-7 means 'never delinquent' -> a GOOD signal).
# Each special code becomes its OWN WoE bin (sparse ones auto-merged), so we can ask:
#   does the stump<->WoE isomorphism and SE alignment hold on INFORMATIVE missing bins?
# Connects to GMSC (MonthlyIncome plain-missing) and German (Foreign_worker sparse level).
# Reuses validated logic (credit WoE, log-odds-limit stump, fair bootstrap, repeated-split AUC).
# Single-cell; greyscale seaborn; dpi600 png+pdf; no captions.

import os, json, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, roc_curve
import statsmodels.api as sm
warnings.filterwarnings('ignore')

SEED=42; N_BOOT=500; N_SPLIT=50
rng=np.random.default_rng(SEED)
sns.set_theme(style='whitegrid', context='paper'); sns.set_palette('Greys')
plt.rcParams.update({'axes.edgecolor':'0.2','grid.color':'0.85','font.size':10})
ROOT=os.path.abspath(os.path.join(os.getcwd(),'..')); DATA=os.path.join(ROOT,'data')
TABLES=os.path.join(ROOT,'results','tables'); FIGS=os.path.join(ROOT,'results','figures')
def savefig(fig,name):
    for e in ('png','pdf'): fig.savefig(os.path.join(FIGS,name+'.'+e),dpi=600,bbox_inches='tight')
    plt.close(fig)
SPECIAL=[-7,-8,-9]

# ===================================================================
# 0. LOAD
# ===================================================================
df=pd.read_csv(os.path.join(DATA,'heloc.csv'))
TARGET='RiskPerformance'
df[TARGET]=(df[TARGET]=='Bad').astype(int)              # bad=1
FEATURES=[c for c in df.columns if c!=TARGET]
# drop the 588 all-(-9) rows (no bureau record at all) -> they carry no feature info
allmiss=(df[FEATURES]==-9).all(axis=1)
df=df[~allmiss].reset_index(drop=True)
y=df[TARGET].values.astype(int); TG=int((y==0).sum()); TB=int((y==1).sum())
print('HELOC:',df.shape,'| bad rate %.3f'%y.mean(),'| dropped %d all-(-9) rows'%int(allmiss.sum()))

# ===================================================================
# 1. WoE BINNING: valid-value monotone bins + per-special-code bins (sparse merged)
# ===================================================================
def woe_of(g,b): return float(np.log(((b+0.5)/(TB+0.5))/((g+0.5)/(TG+0.5))))
MIN_BIN_FRAC=0.01; MIN_N=int(MIN_BIN_FRAC*len(df))

def valid_edges(x,yv,max_leaf=5,min_frac=0.05):
    m=~x.isin(SPECIAL); xx=x[m].values.reshape(-1,1).astype(float); yy=yv[m.values]
    if len(np.unique(xx))<=2: return []
    t=DecisionTreeClassifier(max_leaf_nodes=max_leaf,min_samples_leaf=int(min_frac*len(xx)),random_state=SEED)
    t.fit(xx,yy); thr=t.tree_.threshold[t.tree_.feature>=0]; return sorted(float(v) for v in thr)

def build_bins(x,yv,edges):
    recs=[]; m=~x.isin(SPECIAL)
    cuts=[-np.inf]+list(edges)+[np.inf]; idx=pd.cut(x[m],cuts,labels=False,include_lowest=True)
    for b in range(len(cuts)-1):
        sel=idx==b; g=int((yv[m.values][sel]==0).sum()); bd=int((yv[m.values][sel]==1).sum())
        if g+bd>0: recs.append({'kind':'valid','lo':cuts[b],'hi':cuts[b+1],'good':g,'bad':bd})
    # special-code bins
    for code in SPECIAL:
        sel=x==code; g=int((yv[sel.values]==0).sum()); bd=int((yv[sel.values]==1).sum())
        if g+bd>0: recs.append({'kind':'code%d'%code,'code':code,'good':g,'bad':bd})
    return recs

def monotone_valid(x,yv,edges):
    e=list(edges)
    while e:
        recs=[r for r in build_bins(x,yv,e) if r['kind']=='valid']
        br=np.array([r['bad']/(r['good']+r['bad']) for r in recs])
        if len(br)<=2 or np.all(np.diff(br)>=-1e-12) or np.all(np.diff(br)<=1e-12): return e
        d=np.diff(br); s=np.sign(np.nansum(d)); e.pop(int(np.argmax(-s*d)))
    return e

woe_maps={}; iv_rows=[]
for c in FEATURES:
    edges=monotone_valid(df[c],y,valid_edges(df[c],y)); recs=build_bins(df[c],y,edges)
    # merge sparse special-code bins into a single 'code_other' bin
    keep=[]; sparse_g=sparse_b=0; has_code=False
    for r in recs:
        if r['kind'].startswith('code'):
            has_code=True
            if (r['good']+r['bad'])<MIN_N: sparse_g+=r['good']; sparse_b+=r['bad']; continue
        keep.append(r)
    if sparse_g+sparse_b>0: keep.append({'kind':'code_other','good':sparse_g,'bad':sparse_b})
    for i,r in enumerate(keep): r['bin']=i; r['woe']=woe_of(r['good'],r['bad']); r['n']=r['good']+r['bad']
    woe_maps[c]={'recs':keep,'has_code':has_code}
    iv=sum(((r['bad']+.5)/(TB+.5)-(r['good']+.5)/(TG+.5))*r['woe'] for r in keep)
    n_code=sum(1 for r in keep if r['kind'].startswith('code'))
    iv_rows.append([c,iv,len(keep),n_code])
iv_df=pd.DataFrame(iv_rows,columns=['feature','IV','n_bins','n_code_bins']).sort_values('IV',ascending=False).reset_index(drop=True)
IV_THRESH=0.02
kept=iv_df[iv_df.IV>=IV_THRESH]['feature'].tolist(); dropped=iv_df[iv_df.IV<IV_THRESH]['feature'].tolist()
print('\n=== HELOC IV (top 15) ==='); print(iv_df.head(15).round(4).to_string(index=False))
print('\nKept %d/%d (IV>=%.2f). Dropped %d:'%(len(kept),len(FEATURES),IV_THRESH,len(dropped)),dropped)
FEATURES=[c for c in FEATURES if c in kept]

# ===================================================================
# 2. bin index + WoE
# ===================================================================
def to_bidx(colname):
    recs=woe_maps[colname]['recs']; x=df[colname]; idx=np.full(len(df),-1,int)
    for i,r in enumerate(recs):
        if r['kind']=='valid':
            sel=(~x.isin(SPECIAL)).values&(x>r['lo']).values&(x<=r['hi']).values; idx[sel]=i
        elif r['kind']=='code_other':
            pass
        else:
            idx[(x==r['code']).values]=i
    # assign leftover specials to code_other bin if present
    oth=[i for i,r in enumerate(recs) if r['kind']=='code_other']
    if oth: idx[idx==-1]=oth[0]
    else: idx[idx==-1]=0
    return idx
Xbin={c:to_bidx(c) for c in FEATURES}
bin_woe0={c:np.array([r['woe'] for r in woe_maps[c]['recs']]) for c in FEATURES}
bin_w={c:np.array([r['n'] for r in woe_maps[c]['recs']],float) for c in FEATURES}
is_code_bin={c:np.array([r['kind'].startswith('code') for r in woe_maps[c]['recs']]) for c in FEATURES}
nbins={c:len(bin_woe0[c]) for c in FEATURES}
Xw0=np.column_stack([bin_woe0[c][Xbin[c]] for c in FEATURES])

# ===================================================================
# 3. ISOMORPHISM (log-odds-limit stump vs WoE) + logit beta
# ===================================================================
def logodds_curve(bidx,yv,nb):
    g=np.zeros(nb); b=np.zeros(nb); np.add.at(g,bidx[yv==0],1.0); np.add.at(b,bidx[yv==1],1.0)
    return np.log((b+0.5)/(g+0.5))-np.log((yv==1).sum()/(yv==0).sum())
def beff(s,w,wo):
    d=np.sum(w*wo**2); return float(np.sum(w*s*wo)/d) if d>0 else np.nan
logit0=sm.Logit(y,sm.add_constant(Xw0)).fit(disp=0)
beta_hat={c:logit0.params[j+1] for j,c in enumerate(FEATURES)}
beta_se ={c:logit0.bse[j+1] for j,c in enumerate(FEATURES)}
iso_rows=[]; s_hat={}
for c in FEATURES:
    s=logodds_curve(Xbin[c],y,nbins[c]); s_hat[c]=s; woe=bin_woe0[c]; w=bin_w[c]
    wm_s=np.sum(w*s)/w.sum(); wm_w=np.sum(w*woe)/w.sum()
    cov=np.sum(w*(s-wm_s)*(woe-wm_w)); vs=np.sum(w*(s-wm_s)**2); vw=np.sum(w*(woe-wm_w)**2)
    wcorr=cov/np.sqrt(vs*vw) if vs>0 and vw>0 else np.nan
    iso_rows.append([c,nbins[c],int(is_code_bin[c].sum()),beff(s,w,woe),wcorr,beta_hat[c],beta_se[c]])
iso_df=pd.DataFrame(iso_rows,columns=['feature','n_bins','n_code_bins','beta_eff','w_corr_s_woe','beta','beta_se'])
print('\n=== HELOC isomorphism (ideal regime) ==='); print(iso_df.round(4).to_string(index=False))
print('mean beta_eff=%.4f | mean w_corr=%.4f'%(iso_df.beta_eff.mean(),iso_df.w_corr_s_woe.mean()))

# ===================================================================
# 4. INFORMATIVE-MISSING bin-level analysis: WoE of code bins + s==WoE check
# ===================================================================
code_rows=[]
for c in FEATURES:
    for i,r in enumerate(woe_maps[c]['recs']):
        if r['kind'].startswith('code'):
            code_rows.append([c,r['kind'],r['n'],r['woe'],s_hat[c][i],abs(s_hat[c][i]-r['woe'])])
code_df=pd.DataFrame(code_rows,columns=['feature','code','n','woe','stump_s','abs_diff'])
print('\n=== Informative missing-code bins (WoE sign tells direction) ===')
print(code_df.round(4).to_string(index=False))
print('code-bin s vs WoE: max|diff|=%.4f (isomorphism holds on informative missing bins if ~0)'%code_df.abs_diff.max())

# ===================================================================
# 5. INFERENCE: fair-coordinate bootstrap SE, split by has-code-bin vs not
# ===================================================================
n=len(y); boot_logit={c:[] for c in FEATURES}; boot_stump={c:[] for c in FEATURES}
t0=time.time()
for b in range(N_BOOT):
    idx=rng.integers(0,n,n); yb=y[idx]; tgb=int((yb==0).sum()); tbb=int((yb==1).sum()); cols=[]; wb={}
    for c in FEATURES:
        bi=Xbin[c][idx]; g=np.zeros(nbins[c]); bd=np.zeros(nbins[c]); np.add.at(g,bi[yb==0],1.0); np.add.at(bd,bi[yb==1],1.0)
        w=np.log((bd+0.5)/(tbb+0.5))-np.log((g+0.5)/(tgb+0.5)); wb[c]=w; cols.append(w[bi])
        boot_stump[c].append(np.log((bd+0.5)/(g+0.5))-np.log(tbb/tgb))
    Xwb=np.column_stack(cols)
    try:
        lb=sm.Logit(yb,sm.add_constant(Xwb)).fit(disp=0)
        for j,c in enumerate(FEATURES): boot_logit[c].append(lb.params[j+1]*wb[c])
    except Exception:
        for c in FEATURES: boot_logit[c].append(np.full(nbins[c],np.nan))
    if (b+1)%100==0: print('  bootstrap %d/%d (%.0fs)'%(b+1,N_BOOT,time.time()-t0))
print('bootstrap done %.0fs'%(time.time()-t0))
se_rows=[]
for c in FEATURES:
    L=np.array(boot_logit[c]); S=np.array(boot_stump[c]); sel=np.nanstd(L,0,ddof=1); ses=np.nanstd(S,0,ddof=1)
    w=bin_w[c]; wn=w/w.sum(); se_l=float(np.sqrt(np.sum(wn*sel**2))); se_s=float(np.sqrt(np.sum(wn*ses**2)))
    grp='has_code' if is_code_bin[c].any() else 'no_code'
    se_rows.append([c,grp,se_l,se_s,se_s/se_l if se_l>0 else np.nan])
se_df=pd.DataFrame(se_rows,columns=['feature','group','se_logit','se_stump','se_ratio'])
print('\n=== HELOC SE alignment by group ===')
for g in ['no_code','has_code']:
    v=se_df[se_df.group==g]['se_ratio']
    if len(v): print('  %-9s: mean SE ratio stump/logit = %.3f \u00b1 %.3f (n=%d)'%(g,v.mean(),v.std(),len(v)))

# ===================================================================
# 6. PERFORMANCE: repeated splits logit vs stump (+DeLong)
# ===================================================================
def ks(yt,s): fpr,tpr,_=roc_curve(yt,s); return float(np.max(tpr-fpr))
def delong(yt,s1,s2):
    yt=np.asarray(yt); pos=yt==1; neg=~pos; m=pos.sum(); nn=neg.sum()
    def st(s):
        sp,sn=s[pos],s[neg]; V10=np.array([(np.sum(sn<x)+.5*np.sum(sn==x))/nn for x in sp])
        V01=np.array([(np.sum(sp>x)+.5*np.sum(sp==x))/m for x in sn]); return V10.mean(),V10,V01
    a1,x1,y1=st(s1); a2,x2,y2=st(s2); S=np.cov(np.vstack([x1,x2]))/m+np.cov(np.vstack([y1,y2]))/nn
    v=S[0,0]+S[1,1]-2*S[0,1]
    if v<=0: return a1,a2,np.nan,np.nan
    z=(a1-a2)/np.sqrt(v); return a1,a2,float(z),float(2*(1-stats.norm.cdf(abs(z))))
perf=[]; dl=[]
for sp in range(N_SPLIT):
    Xtr,Xte,ytr,yte=train_test_split(Xw0,y,test_size=0.3,stratify=y,random_state=SEED+sp)
    lr=LogisticRegression(max_iter=1000).fit(Xtr,ytr); s_lr=lr.decision_function(Xte)
    gb=GradientBoostingClassifier(max_depth=1,learning_rate=0.3,n_estimators=300,random_state=SEED+sp).fit(Xtr,ytr)
    s_gb=gb.decision_function(Xte)
    for nm,s in [('logit_woe',s_lr),('stump_boost',s_gb)]: perf.append([sp,nm,roc_auc_score(yte,s),ks(yte,s)])
    a_gb,a_lr,z,p=delong(yte,s_gb,s_lr); dl.append([sp,a_gb-a_lr,z,p])
perf=pd.DataFrame(perf,columns=['split','model','auc','ks']); dl=pd.DataFrame(dl,columns=['split','auc_diff','z','p'])
aucl=perf[perf.model=='logit_woe'].auc; aucs=perf[perf.model=='stump_boost'].auc
w_sl=stats.wilcoxon(aucs.values,aucl.values)
print('\n=== HELOC performance (%d splits) ==='%N_SPLIT)
print('logit AUC=%.4f\u00b1%.4f | stump AUC=%.4f\u00b1%.4f | diff=%+.5f Wilcoxon p=%.4f'%(
    aucl.mean(),aucl.std(),aucs.mean(),aucs.std(),aucs.mean()-aucl.mean(),w_sl.pvalue))

# ===================================================================
# 7. SAVE + FIGURES
# ===================================================================
iv_df.round(6).to_csv(os.path.join(TABLES,'06_heloc_iv.csv'),index=False)
iso_df.round(6).to_csv(os.path.join(TABLES,'06_heloc_isomorphism.csv'),index=False)
code_df.round(6).to_csv(os.path.join(TABLES,'06_heloc_code_bins.csv'),index=False)
se_df.round(6).to_csv(os.path.join(TABLES,'06_heloc_se.csv'),index=False)
perf.round(6).to_csv(os.path.join(TABLES,'06_heloc_perf.csv'),index=False)

# Fig A: s vs WoE, highlighting informative-missing code bins
allw=np.concatenate([bin_woe0[c] for c in FEATURES]); alls=np.concatenate([s_hat[c] for c in FEATURES])
codemask=np.concatenate([is_code_bin[c] for c in FEATURES])
fig,ax=plt.subplots(figsize=(5.8,5.6)); lim=np.array([min(allw.min(),alls.min()),max(allw.max(),alls.max())])*1.05
ax.plot(lim,lim,color='0.0',lw=1,ls='--')
ax.scatter(allw[~codemask],alls[~codemask],s=30,marker='o',c='0.45',edgecolor='0.1',lw=.3,label='valid-value bins')
ax.scatter(allw[codemask],alls[codemask],s=70,marker='D',c='0.0',edgecolor='0.0',lw=.5,label='informative-missing bins')
r=np.corrcoef(allw,alls)[0,1]; ax.set_title('HELOC: s(bin) vs WoE(bin)  (r=%.3f)'%r,fontsize=10)
ax.set_xlabel('WoE(bin)'); ax.set_ylabel('stump s(bin)'); ax.set_xlim(lim); ax.set_ylim(lim); ax.set_aspect('equal')
ax.legend(frameon=True,facecolor='white',edgecolor='0.5'); savefig(fig,'06_heloc_s_vs_woe')

# Fig B: SE ratio by group (strip)
fig,ax=plt.subplots(figsize=(5.6,4.4))
for i,(g,cl) in enumerate([('no_code','0.35'),('has_code','0.7')]):
    v=se_df[se_df.group==g]['se_ratio'].values; x=np.random.default_rng(i).normal(i,0.05,len(v))
    ax.scatter(x,v,s=45,c=cl,edgecolor='0.0',lw=.4,label='%s (n=%d)'%(g,len(v)))
ax.axhline(1.0,color='0.0',lw=1,ls='--'); ax.set_xticks([0,1]); ax.set_xticklabels(['no code bin','has code bin'])
ax.set_ylabel('SE ratio stump/logit'); ax.set_title('HELOC SE alignment by missing-bin presence',fontsize=10)
ax.legend(frameon=True,facecolor='white',edgecolor='0.5'); savefig(fig,'06_heloc_se_by_group')

print('\nSaved figures: 06_heloc_s_vs_woe, 06_heloc_se_by_group')
print('Saved tables : 06_heloc_iv, 06_heloc_isomorphism, 06_heloc_code_bins, 06_heloc_se, 06_heloc_perf')

# ===================================================================
# 8. THREE-DATASET HEADLINE
# ===================================================================
print('\n================ HELOC (informative-missing) HEADLINE ================')
print('isomorphism: mean beta_eff=%.3f, mean w_corr=%.3f'%(iso_df.beta_eff.mean(),iso_df.w_corr_s_woe.mean()))
print('informative-missing code bins: %d bins, max|s-WoE|=%.4f (isomorphism holds on them)'%(len(code_df),code_df.abs_diff.max()))
nc=se_df[se_df.group=='no_code']['se_ratio']; hc=se_df[se_df.group=='has_code']['se_ratio']
print('SE ratio: no_code=%.3f vs has_code=%.3f'%(nc.mean(),hc.mean()))
print('performance: logit AUC=%.3f vs stump AUC=%.3f (p=%.3f)'%(aucl.mean(),aucs.mean(),w_sl.pvalue))
print('Cross-dataset: GMSC beta_eff~1.00, German~1.00, HELOC~%.2f => isomorphism universal.'%iso_df.beta_eff.mean())
print('\n06 complete.')


HELOC: (9871, 24) | bad rate 0.520 | dropped 588 all-(-9) rows



=== HELOC IV (top 15) ===
                           feature     IV  n_bins  n_code_bins
              ExternalRiskEstimate 0.9784       6            1
        NetFractionRevolvingBurden 0.5666       5            1
NumBank2NatlTradesWHighUtilization 0.3621       6            1
            PercentTradesNeverDelq 0.3621       5            0
             PercentTradesWBalance 0.3430       6            1
      MSinceMostRecentInqexcl7days 0.3237       6            2
          MaxDelq2PublicRecLast12M 0.3202       3            0
                    AverageMInFile 0.3069       5            0
              MSinceMostRecentDelq 0.3065       7            2
             MSinceOldestTradeOpen 0.2309       6            1
                       MaxDelqEver 0.2299       4            0
       NumTrades60Ever2DerogPubRec 0.1821       4            0
       NumTrades90Ever2DerogPubRec 0.1327       3            0
             NumSatisfactoryTrades 0.1233       5            0
        NumRevolvingTradesWB

  bootstrap 100/500 (8s)


  bootstrap 200/500 (16s)


  bootstrap 300/500 (23s)


  bootstrap 400/500 (32s)


  bootstrap 500/500 (39s)
bootstrap done 39s

=== HELOC SE alignment by group ===
  no_code  : mean SE ratio stump/logit = 0.786 ± 0.330 (n=12)
  has_code : mean SE ratio stump/logit = 1.231 ± 0.226 (n=9)



=== HELOC performance (50 splits) ===
logit AUC=0.8086±0.0069 | stump AUC=0.8061±0.0068 | diff=-0.00245 Wilcoxon p=0.0000



Saved figures: 06_heloc_s_vs_woe, 06_heloc_se_by_group
Saved tables : 06_heloc_iv, 06_heloc_isomorphism, 06_heloc_code_bins, 06_heloc_se, 06_heloc_perf

================ HELOC (informative-missing) HEADLINE ================
isomorphism: mean beta_eff=1.000, mean w_corr=1.000
informative-missing code bins: 11 bins, max|s-WoE|=0.0000 (isomorphism holds on them)
SE ratio: no_code=0.786 vs has_code=1.231
performance: logit AUC=0.809 vs stump AUC=0.806 (p=0.000)
Cross-dataset: GMSC beta_eff~1.00, German~1.00, HELOC~1.00 => isomorphism universal.

06 complete.
